## 1. Setup & Dependencies

In [ ]:
# Check if running in Colab
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    print("✅ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("ℹ️  Running locally")

In [ ]:
# Install dependencies (only needed in Colab)
if IN_COLAB:
    !pip install -q datasets pillow music21 python-dotenv tqdm matplotlib
    !pip install -q oemer onnxruntime-gpu  # Latest version with CUDA 12 support
    print("✅ Dependencies installed")
else:
    print("ℹ️  Using local environment - ensure dependencies are installed")

In [ ]:
# GPU Check
import subprocess

try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                          capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ GPU Available:")
        print(result.stdout.strip())
    else:
        print("⚠️  No GPU detected - using CPU")
except Exception:
    print("⚠️  nvidia-smi not available - using CPU")

## 2. HuggingFace Authentication

In [ ]:
# Set HuggingFace token (REQUIRED for PRAIG/SMB dataset)
import os
from getpass import getpass

if 'HF_TOKEN' not in os.environ:
    print("🔑 HuggingFace Token Required")
    print("Get yours at: https://huggingface.co/settings/tokens")
    print("Request access: https://huggingface.co/datasets/PRAIG/SMB")
    hf_token = getpass("Enter HF_TOKEN: ")
    os.environ['HF_TOKEN'] = hf_token
    print("✅ Token set")
else:
    print("✅ HF_TOKEN already configured")

## 3. Load Dataset

In [ ]:
from datasets import load_dataset
from PIL import Image
import io

print("📥 Loading PRAIG/SMB dataset...")
dataset = load_dataset("PRAIG/SMB", split="test", token=os.environ['HF_TOKEN'])
print(f"✅ Loaded {len(dataset)} samples")

In [ ]:
# Helper function to format dataset items
def format_sample(item):
    """Extract image and ground truth from dataset item."""
    # Get image
    image = item['image']
    if not isinstance(image, Image.Image):
        image = Image.open(io.BytesIO(image)).convert('L')
    
    # Extract ground truth from regions
    ground_truth = ""
    if 'regions' in item and item['regions']:
        for region in item['regions']:
            if 'kern' in region and region['kern']:
                ground_truth += region['kern'] + "\n"
    
    # Fallback to page-level kern if regions empty
    if not ground_truth and 'kern' in item:
        ground_truth = item['kern'] or ""
    
    return {
        'image': image,
        'ground_truth': ground_truth.strip(),
        'filename': item.get('filename', 'unknown')
    }

print("✅ Helper functions ready")

## 4. Dataset Explorer

In [ ]:
import matplotlib.pyplot as plt
import random

def show_sample(idx):
    """Display a sample from the dataset."""
    sample = format_sample(dataset[idx])
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Image
    ax1.imshow(sample['image'], cmap='gray')
    ax1.axis('off')
    ax1.set_title(f"Sample {idx}: {sample['filename']}\nSize: {sample['image'].size}")
    
    # Ground truth
    gt = sample['ground_truth']
    ax2.text(0.05, 0.95, gt[:1000] + ("\n\n[...]" if len(gt) > 1000 else ""),
             verticalalignment='top', fontfamily='monospace', fontsize=8, wrap=True)
    ax2.axis('off')
    ax2.set_title(f"Ground Truth ({len(gt)} chars)")
    
    plt.tight_layout()
    plt.show()
    
    return sample

# Show random sample
sample_idx = random.randint(0, len(dataset)-1)
sample = show_sample(sample_idx)

## 5. Load OeMeR Model

In [ ]:
import tempfile
import subprocess
from pathlib import Path

def predict_with_oemer(image, debug=False):
    """Run OeMeR on an image and return MusicXML."""
    with tempfile.TemporaryDirectory() as temp_dir:
        temp_dir = Path(temp_dir)
        
        # Save image
        input_path = temp_dir / "input.png"
        image.save(input_path)
        
        # Run OeMeR
        output_dir = temp_dir / "output"
        output_dir.mkdir()
        
        try:
            result = subprocess.run(
                ["oemer", str(input_path), "-o", str(output_dir)],
                capture_output=True,
                text=True,
                timeout=480  # 8 minutes
            )
            
            if result.returncode != 0:
                print(f"❌ OeMeR failed: {result.stderr}")
                return None
            
            # Find output MusicXML
            musicxml_files = list(output_dir.glob("*.musicxml"))
            if not musicxml_files:
                print("❌ No MusicXML output found")
                return None
            
            with open(musicxml_files[0], 'r', encoding='utf-8') as f:
                return f.read()
                
        except subprocess.TimeoutExpired:
            print("❌ OeMeR timeout (8 min)")
            return None
        except Exception as e:
            print(f"❌ Error: {e}")
            return None

print("✅ OeMeR wrapper ready")

In [ ]:
# MusicXML to **kern converter (simplified)
from music21 import converter, note, chord

def musicxml_to_kern(musicxml_str):
    """Convert MusicXML to **kern notation (simplified)."""
    try:
        score = converter.parse(musicxml_str)
        
        def pitch_to_kern(pitch):
            """Convert music21 pitch to **kern."""
            name = pitch.name.replace('-', 'b').replace('#', '#')
            octave = pitch.octave
            
            if octave >= 4:
                return name.lower() * (octave - 3)
            else:
                return name.upper() * (4 - octave)
        
        def duration_to_kern(duration):
            """Convert duration to **kern."""
            reciprocal = int(1.0 / duration.quarterLength) if duration.quarterLength > 0 else 4
            return str(reciprocal)
        
        # Extract all parts
        parts = score.parts
        if not parts:
            return ""
        
        # Build **kern for each part
        kern_parts = []
        for part in parts:
            kern_output = ["**kern"]
            
            for element in part.flatten().notesAndRests:
                dur = duration_to_kern(element.duration)
                
                if isinstance(element, note.Note):
                    kern_output.append(dur + pitch_to_kern(element.pitch))
                elif isinstance(element, chord.Chord):
                    pitches = ' '.join(pitch_to_kern(p) for p in element.pitches)
                    kern_output.append(dur + pitches)
                elif isinstance(element, note.Rest):
                    kern_output.append(dur + 'r')
            
            kern_output.append("*-")
            kern_parts.append(' '.join(kern_output))
        
        # Join multiple staves with TAB
        return '\t'.join(kern_parts)
        
    except Exception as e:
        print(f"❌ Conversion error: {e}")
        return ""

print("✅ Converter ready")

## 6. Test Single Prediction

In [ ]:
# Test on a sample
print("🎵 Running OeMeR prediction...")
print("(First run takes 2-5 min for model compilation)\n")

musicxml = predict_with_oemer(sample['image'])
if musicxml:
    print(f"✅ MusicXML generated ({len(musicxml)} chars)")
    
    kern = musicxml_to_kern(musicxml)
    print(f"✅ **kern converted ({len(kern)} chars)")
    print("\nPrediction (first 500 chars):")
    print(kern[:500])
    print("\nGround Truth (first 500 chars):")
    print(sample['ground_truth'][:500])
else:
    print("❌ Prediction failed")

## 7. Metrics Calculation

In [ ]:
import numpy as np

def levenshtein_distance(s1, s2):
    """Calculate Levenshtein distance between two strings."""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    
    if len(s2) == 0:
        return len(s1)
    
    previous_row = np.arange(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]

def calculate_ned(prediction, ground_truth):
    """Calculate Normalized Edit Distance (OMR-NED)."""
    if not ground_truth:
        return 1.0 if prediction else 0.0
    
    distance = levenshtein_distance(prediction, ground_truth)
    return distance / len(ground_truth)

# Test metrics
if 'kern' in locals() and kern:
    ned = calculate_ned(kern, sample['ground_truth'])
    print(f"\n📊 OMR-NED Score: {ned:.4f}")
    print("   (Lower is better, 0.0 = perfect match)")
else:
    print("⚠️  Run prediction first")

## 8. Batch Evaluation

In [ ]:
from tqdm.notebook import tqdm
import pandas as pd

# Configuration
NUM_SAMPLES = 10  # Change this to evaluate more samples
print(f"🔬 Evaluating {NUM_SAMPLES} samples...\n")

results = []

for idx in tqdm(range(NUM_SAMPLES)):
    sample = format_sample(dataset[idx])
    
    # Predict
    musicxml = predict_with_oemer(sample['image'])
    if not musicxml:
        results.append({
            'idx': idx,
            'filename': sample['filename'],
            'ned': None,
            'error': 'Prediction failed'
        })
        continue
    
    # Convert
    kern = musicxml_to_kern(musicxml)
    
    # Calculate NED
    ned = calculate_ned(kern, sample['ground_truth'])
    
    results.append({
        'idx': idx,
        'filename': sample['filename'],
        'ned': ned,
        'pred_len': len(kern),
        'gt_len': len(sample['ground_truth']),
        'error': None
    })

# Create DataFrame
df = pd.DataFrame(results)
print("\n✅ Evaluation complete!")
df.head(10)

## 9. Results Analysis

In [ ]:
# Statistics
valid_neds = df[df['ned'].notna()]['ned']

print("=" * 60)
print("📊 Benchmark Results")
print("=" * 60)
print(f"Total Samples:       {len(df)}")
print(f"Successful:          {len(valid_neds)} ({len(valid_neds)/len(df)*100:.1f}%)")
print(f"Failed:              {len(df) - len(valid_neds)}")
print("\nOMR-NED Statistics (lower is better):")
print(f"  Mean:              {valid_neds.mean():.4f}")
print(f"  Median:            {valid_neds.median():.4f}")
print(f"  Std Dev:           {valid_neds.std():.4f}")
print(f"  Min (Best):        {valid_neds.min():.4f}")
print(f"  Max (Worst):       {valid_neds.max():.4f}")
print(f"\nPerfect Matches:     {(valid_neds == 0).sum()} ({(valid_neds == 0).sum()/len(valid_neds)*100:.1f}%)")
print("=" * 60)

In [ ]:
# Visualization

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NED distribution
axes[0].hist(valid_neds, bins=20, edgecolor='black', alpha=0.7)
axes[0].axvline(valid_neds.mean(), color='red', linestyle='--', label=f'Mean: {valid_neds.mean():.4f}')
axes[0].set_xlabel('OMR-NED Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('NED Score Distribution')
axes[0].legend()
axes[0].grid(alpha=0.3)

# NED over samples
valid_df = df[df['ned'].notna()]
axes[1].plot(valid_df['idx'], valid_df['ned'], marker='o', linestyle='-', alpha=0.6)
axes[1].axhline(valid_neds.mean(), color='red', linestyle='--', label='Mean')
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('OMR-NED Score')
axes[1].set_title('NED Scores by Sample')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Export Results

In [ ]:
import json
from datetime import datetime

# Export to JSON
output = {
    'timestamp': datetime.now().isoformat(),
    'model': 'OeMeR',
    'dataset': 'PRAIG/SMB',
    'num_samples': len(df),
    'statistics': {
        'mean_ned': float(valid_neds.mean()),
        'median_ned': float(valid_neds.median()),
        'std_ned': float(valid_neds.std()),
        'min_ned': float(valid_neds.min()),
        'max_ned': float(valid_neds.max()),
        'perfect_matches': int((valid_neds == 0).sum())
    },
    'results': df.to_dict(orient='records')
}

output_file = f"benchmark_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

if IN_COLAB:
    # Save to Colab files
    with open(output_file, 'w') as f:
        json.dump(output, f, indent=2)
    print(f"✅ Results saved to: {output_file}")
    print("   Download from Files panel (left sidebar)")
else:
    # Save locally
    with open(output_file, 'w') as f:
        json.dump(output, f, indent=2)
    print(f"✅ Results saved to: {output_file}")

# Also export CSV
csv_file = output_file.replace('.json', '.csv')
df.to_csv(csv_file, index=False)
print(f"✅ CSV saved to: {csv_file}")

## 11. Best & Worst Samples

In [ ]:
# Show best and worst performing samples
valid_df = df[df['ned'].notna()].copy()

if len(valid_df) > 0:
    best_idx = valid_df.loc[valid_df['ned'].idxmin(), 'idx']
    worst_idx = valid_df.loc[valid_df['ned'].idxmax(), 'idx']
    
    print("🏆 Best Performance:")
    show_sample(int(best_idx))
    
    print("\n⚠️  Worst Performance:")
    show_sample(int(worst_idx))
else:
    print("⚠️  No valid results to display")